# Wind Farm Simulation with FLORIS

In this tutorial you will use [FLORIS](https://natlabrockies.github.io/floris/), an open-source wind farm simulation framework ({cite}`floris`), to see how turbines affect each other through their **wakes**. A turbine extracts energy from the wind, so the air behind it is slower and more turbulent. Downstream turbines therefore produce less power than turbines in undisturbed flow.

After this tutorial you will be able to:

1. create a FLORIS model and define a wind farm layout;
2. set the inflow conditions (wind speed, wind direction, turbulence intensity);
3. run the simulation and read the power of each turbine and of the whole farm;
4. visualise the wake flow field and explain how the wind direction changes the farm power.

This tutorial is adapted from the FLORIS examples [001](https://natlabrockies.github.io/floris/examples/001_opening_floris_computing_power.html) and [002](https://natlabrockies.github.io/floris/examples/002_visualizations.html).

```{tip} Interactive
To run the code, click the ![](figures/on.png) icon in the top right corner to connect to a kernel. Once the kernel is loaded, click ![](figures/play.png) to run a cell. Run the cells **in order**, from top to bottom.
```

## 1. Set up the model

The `FlorisModel` class is the entry point for most FLORIS usage. Initialising it with `"defaults"` loads default wake model and atmospheric settings. We then select the IEA 10 MW reference turbine from the FLORIS turbine library (198 m rotor diameter, 119 m hub height) and place 8 turbines on an irregular layout, given as lists of $x$ and $y$ coordinates in metres.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact

import floris.layout_visualization as layoutviz
from floris import FlorisModel
from floris.flow_visualization import visualize_cut_plane

fmodel = FlorisModel("defaults")

# Turbine positions [m]
layout_x = [0, 500, 0, 128, 1000, 900, 1500, 1250]
layout_y = [0, 300, 750, 1400, 0, 567, 888, 1450]
fmodel.set(layout_x=layout_x, layout_y=layout_y, turbine_type=["iea_10MW"])
# Apply the wind speed at the turbine hub height
fmodel.assign_hub_height_to_ref_height()

rotor_diameter = fmodel.core.farm.rotor_diameters.flatten()[0]
hub_height = fmodel.core.farm.hub_heights.flatten()[0]

# Plot style: accent colour palette, no grid
from cycler import cycler
plt.rcParams.update({
    'axes.prop_cycle': cycler(color=['#017188', '#EC6842', '#00A6D6', '#6CC24A', '#A50034']),
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 2,
})

: 

Let's check the layout. The numbers are the turbine indices, which we will use later to read the power per turbine.

In [ ]:
fig, ax = plt.subplots()
layoutviz.plot_turbine_points(fmodel, ax=ax)
layoutviz.plot_turbine_labels(fmodel, ax=ax)
ax.set_title("Wind farm layout")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal")
plt.show()

## 2. Run a single wind condition

The inflow is defined by three quantities:

- the **wind speed** at hub height [m/s];
- the **wind direction** [deg], defined as the direction the wind comes *from* (meteorological convention): $270^\circ$ is a westerly wind, blowing towards the east (+$x$), and $0^\circ$ is a northerly wind, blowing towards the south;
- the **turbulence intensity** [-], which controls how fast the wake recovers.

Each is given as a list: FLORIS can evaluate many conditions at once (one per list entry). Here we use a single condition.

In [ ]:
fmodel.set(wind_speeds=[8.0], wind_directions=[270.0], turbulence_intensities=[0.06])
fmodel.run()

# Arrays have shape (number of conditions, number of turbines); power is in W
turbine_powers = fmodel.get_turbine_powers() / 1e6
farm_power = fmodel.get_farm_power() / 1e6

for i, p in enumerate(turbine_powers[0]):
    print(f"Turbine {i}: {p:.2f} MW")
print(f"Farm power: {farm_power[0]:.2f} MW")

```{admonition} Question
:class: note
Which turbines produce the least power for a westerly wind? Compare with the layout plot above and explain why.
```

## 3. Visualise the flow field

To visualise the wakes we ask FLORIS to compute the wind speed on a horizontal plane at hub height (119 m for the IEA 10 MW turbine). The resolution is kept low so that the calculation is fast.

In [ ]:
horizontal_plane = fmodel.calculate_horizontal_plane(
    x_resolution=100, y_resolution=100, height=hub_height
)

fig, ax = plt.subplots()
visualize_cut_plane(
    horizontal_plane,
    ax=ax,
    label_contours=False,
    title="Wind speed at hub height (wind from the west)",
    cmap="bone",
)
layoutviz.plot_turbine_rotors(fmodel, ax=ax)
layoutviz.plot_turbine_labels(fmodel, ax=ax)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
plt.show()

## 4. Interactive: effect of the wind direction

Use the sliders to change the wind direction and the wind speed. The farm power is shown in the plot title. The calculation is only updated when you release the slider.

In [ ]:
def update(wind_direction, wind_speed):
    fmodel.set(
        wind_speeds=[wind_speed],
        wind_directions=[wind_direction],
        turbulence_intensities=[0.06],
    )
    fmodel.run()
    farm_power = fmodel.get_farm_power()[0] / 1e6

    horizontal_plane = fmodel.calculate_horizontal_plane(
        x_resolution=50, y_resolution=50, height=hub_height
    )

    fig, ax = plt.subplots()
    visualize_cut_plane(
        horizontal_plane,
        ax=ax,
        label_contours=False,
        title=f"Wind from {wind_direction:.0f}$^\\circ$, {wind_speed:.1f} m/s: farm power = {farm_power:.2f} MW",
        cmap="bone",
    )
    layoutviz.plot_turbine_rotors(fmodel, ax=ax)
    layoutviz.plot_turbine_labels(fmodel, ax=ax)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_xlim([-500, 2000])
    ax.set_ylim([-500, 2000])
    ax.set_aspect("equal", adjustable="box")
    plt.show()


interact(
    update,
    wind_direction=widgets.FloatSlider(
        min=0, max=360, step=30, value=270, description="Direction [deg]",
        continuous_update=False, style={"description_width": "initial"},
    ),
    wind_speed=widgets.FloatSlider(
        min=4, max=14, step=1, value=8, description="Speed [m/s]",
        continuous_update=False, style={"description_width": "initial"},
    ),
);

## 5. Farm power versus wind direction

Instead of moving the slider one value at a time, we can pass many conditions to FLORIS in a single run. Here we sweep the wind direction from $0^\circ$ to $359^\circ$ at constant wind speed.

In [ ]:
wind_directions = np.arange(0, 360, 1.0)
n = len(wind_directions)

fmodel.set(
    wind_speeds=8.0 * np.ones(n),
    wind_directions=wind_directions,
    turbulence_intensities=0.06 * np.ones(n),
)
fmodel.run()
farm_power = fmodel.get_farm_power() / 1e6

fig, ax = plt.subplots()
ax.plot(wind_directions, farm_power)
ax.set_xlabel("Wind direction [deg]")
ax.set_ylabel("Farm power [MW]")
ax.set_title("Farm power at 8 m/s")
plt.show()